In [ ]:
# !pip install twikit

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.9/82.9 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.8/611.8 kB 26.0 MB/s eta 0:00:00
  Created wheel for pyjsparser: filename=pyjsparser-2.7.1-py3-none-any.whl size=25982 sha256=86eb8ecb34690fd97ad94c067855ce7bd811f0feff29da6a0f2f6afe8489d646
  Stored in directory: /root/.cache/pip/wheels/14/32/1d/9ef7b582e358446aeef4b9052aa89ef4dffa1688c1aae8aa13
Successfully built pyjsparser


In [ ]:
import json
import ssl
import pandas as pd
import pytz
import os
import asyncio
import threading
import signal
import sys
import csv
from datetime import datetime
from websocket import WebSocketApp
from twikit import Client, TooManyRequests

In [ ]:
# ==========================================================
# CONFIG
# ==========================================================
IST = pytz.timezone("Asia/Kolkata")

TWEET_INTERVAL = 60
TWEET_CSV = "live_btc_tweets.csv"
CANDLE_CSV = "btc_1m_candles.csv"

WS_URL = "wss://stream.bybit.com/v5/public/spot"
QUERY = "(btc OR bitcoin OR BTCUSDT) lang:en -is:retweet"

last_seen_id = None
last_saved_timestamp = None

# ==========================================================
# CREATE FILES IF NOT EXISTS
# ==========================================================
if not os.path.exists(TWEET_CSV):
    pd.DataFrame(columns=[
        "tweet_id",
        "Timestamp",
        "followers",
        "text"
    ]).to_csv(TWEET_CSV, index=False)

if not os.path.exists(CANDLE_CSV):
    pd.DataFrame(columns=[
        "Timestamp",
        "open",
        "high",
        "low",
        "close",
        "volume"
    ]).to_csv(CANDLE_CSV, index=False)

# ==========================================================
# RESTORE STATE FROM EXISTING FILES
# ==========================================================
if os.path.exists(TWEET_CSV):
    df_existing = pd.read_csv(
        TWEET_CSV,
        engine="python",
        on_bad_lines="skip"
    )

    if not df_existing.empty and "tweet_id" in df_existing.columns:
        last_seen_id = df_existing["tweet_id"].max()
    else:
        print("tweet_id column missing. Starting fresh.")
        last_seen_id = None

# ==========================================================
# TWITTER LOGIN
# ==========================================================
client = Client("en-US")
client.set_cookies({
    "auth_token": "7ff2074bbb31051133c6110d82ac9c67d498d07b",
    "ct0": "19f62c63e61204b687fe27c013ea84cdcddc668275a38dfcc39ab35323f14574fe827ed3033bfc71dc9179b6e9a0bd5c91d74f5689e52cca2d888e04531ec53d2671fa70eef802f8f67d142125241a1c"
})
print("Twitter logged in.")

# ==========================================================
# TWEET FETCHER
# ==========================================================
async def fetch_tweets():
    global last_seen_id

    while True:
        try:
            tweets = await client.search_tweet(QUERY, product="Latest")
            rows = []

            for t in tweets:
                if last_seen_id and int(t.id) <= int(last_seen_id):
                    continue

                ts = (
                  pd.to_datetime(t.created_at, utc=True)
                  .tz_convert(IST)
                  .floor("min")          # align to minute
                  .tz_localize(None)     # 🔥 REMOVE timezone
              )

                rows.append({
                    "tweet_id": int(t.id),
                    "Timestamp": ts.strftime("%Y-%m-%d %H:%M:%S%z"),  # SAFE timezone storage
                    "followers": int(t.user.followers_count),
                    "text": t.text.replace("\n", " ").replace("\r", " ")
                })

            if rows:
                df = pd.DataFrame(rows)

                df.to_csv(
                    TWEET_CSV,
                    mode="a",
                    header=False,
                    index=False,
                    quoting=csv.QUOTE_ALL  # Prevent comma corruption
                )

                last_seen_id = max(r["tweet_id"] for r in rows)
                print(f"[Tweets] Saved {len(rows)} new tweets")

        except TooManyRequests as e:
            reset = datetime.fromtimestamp(e.rate_limit_reset)
            wait = max((reset - datetime.now()).seconds, 60)
            print(f"[Tweets] Rate limit. Sleeping {wait}s")
            await asyncio.sleep(wait)

        except Exception as e:
            print("[Tweets] Error:", e)

        await asyncio.sleep(TWEET_INTERVAL)

# ==========================================================
# CANDLE STREAM
# ==========================================================
def on_message(ws, message):
    global last_saved_timestamp

    data = json.loads(message)

    if "topic" in data and data["topic"].startswith("kline.1"):

        k = data["data"][0]

        if not k["confirm"]:
            return

        ts = (
          pd.to_datetime(k['timestamp'], unit='ms', utc=True)
          .tz_convert(IST)
          .floor("min")          # align to minute
          .tz_localize(None)     # 🔥 REMOVE timezone
      )

        # HARD duplicate protection
        if last_saved_timestamp and ts <= last_saved_timestamp:
            return

        candle = {
            "Timestamp": ts.strftime("%Y-%m-%d %H:%M:%S"),  # Store ISO format
            "open": float(k["open"]),
            "high": float(k["high"]),
            "low": float(k["low"]),
            "close": float(k["close"]),
            "volume": float(k["volume"]),
        }

        pd.DataFrame([candle]).to_csv(
            CANDLE_CSV,
            mode="a",
            header=False,
            index=False
        )

        last_saved_timestamp = ts
        print("[Candle] Saved:", ts)


def on_open(ws):
    print("Bybit connected.")
    sub = {"op": "subscribe", "args": ["kline.1.BTCUSDT"]}
    ws.send(json.dumps(sub))


def on_close(ws, *args):
    print("Bybit disconnected.")


def on_error(ws, error):
    print("[Candle] Error:", error)


def start_socket():
    ws = WebSocketApp(
        WS_URL,
        on_open=on_open,
        on_message=on_message,
        on_close=on_close,
        on_error=on_error
    )
    ws.run_forever(sslopt={"cert_reqs": ssl.CERT_NONE})

# ==========================================================
# MERGE FUNCTION
# ==========================================================
def merge_data():
    print("Merging datasets...")

    tweets = pd.read_csv(TWEET_CSV)
    candles = pd.read_csv(CANDLE_CSV)

    # Proper timezone restoration
    tweets["Timestamp"] = pd.to_datetime(tweets["Timestamp"], utc=True)
    tweets["Timestamp"] = tweets["Timestamp"].dt.tz_convert(IST)

    candles["Timestamp"] = pd.to_datetime(candles["Timestamp"], utc=True)
    candles["Timestamp"] = candles["Timestamp"].dt.tz_convert(IST)

    tweets.set_index("Timestamp", inplace=True)
    candles.set_index("Timestamp", inplace=True)

# ==========================================================
# CLEAN EXIT
# ==========================================================
def shutdown(sig, frame):
    print("\nShutting down...")
    merge_data()
    sys.exit(0)

signal.signal(signal.SIGINT, shutdown)

# ==========================================================
# START BOTH SYSTEMS (Notebook Compatible)
# ==========================================================
def start_system():
    print("Starting Tweet + Candle system...\n")

    ws_thread = threading.Thread(target=start_socket)
    ws_thread.daemon = True
    ws_thread.start()

    loop = asyncio.get_event_loop()
    loop.create_task(fetch_tweets())

start_system()
await asyncio.sleep(10**10)


tweet_id column missing. Starting fresh.
Twitter logged in.
Starting Tweet + Candle system...

Bybit connected.
[Tweets] Saved 17 new tweets
[Candle] Saved: 2026-02-23 14:08:00
[Tweets] Saved 19 new tweets
[Candle] Saved: 2026-02-23 14:09:00
[Tweets] Saved 17 new tweets
[Candle] Saved: 2026-02-23 14:10:00
[Tweets] Saved 17 new tweets
[Candle] Saved: 2026-02-23 14:11:00
[Tweets] Saved 19 new tweets
[Candle] Saved: 2026-02-23 14:12:00
[Tweets] Saved 17 new tweets
[Candle] Saved: 2026-02-23 14:13:00
[Tweets] Saved 15 new tweets
[Candle] Saved: 2026-02-23 14:14:00
[Tweets] Saved 16 new tweets
[Candle] Saved: 2026-02-23 14:15:00
[Tweets] Saved 14 new tweets
[Candle] Saved: 2026-02-23 14:16:00
[Tweets] Saved 17 new tweets
[Candle] Saved: 2026-02-23 14:17:00
[Tweets] Saved 15 new tweets
[Candle] Saved: 2026-02-23 14:18:00
[Tweets] Saved 17 new tweets
[Candle] Saved: 2026-02-23 14:19:00
[Tweets] Saved 16 new tweets
[Candle] Saved: 2026-02-23 14:20:00
[Tweets] Saved 19 new tweets
[Candle] Saved

In [ ]:
import pandas as pd

In [ ]:
historical = pd.read_csv('btc_1m_candles.csv')
historical.shape

(42, 6)

In [ ]:
historical

,Timestamp,open,high,low,close,volume
0,2026-02-23 14:08:00,65631.8,65665.5,65600.5,65646.0,32.178420
1,2026-02-23 14:09:00,65646.0,65666.3,65583.8,65584.6,39.064071
2,2026-02-23 14:10:00,65584.6,65630.1,65584.5,65623.8,10.141517
3,2026-02-23 14:11:00,65623.8,65625.4,65607.3,65614.6,3.307413
4,2026-02-23 14:12:00,65614.6,65623.5,65606.5,65623.5,2.225538
5,2026-02-23 14:13:00,65623.5,65623.5,65600.1,65600.2,2.995936
6,2026-02-23 14:14:00,65600.2,65600.2,65584.5,65599.0,4.260788
7,2026-02-23 14:15:00,65599.0,65629.0,65598.9,65619.6,4.933960
8,2026-02-23 14:16:00,65619.6,65643.8,65613.1,65613.1,5.177399
9,2026-02-23 14:17:00,65613.1,65631.3,65606.8,65606.8,4.303491


In [ ]:
tweets = pd.read_csv(
    "live_btc_tweets.csv",
    engine="python",
    on_bad_lines="skip"
)
tweets.shape

(743, 4)

In [ ]:
tweets

,tweet_id,Timestamp,followers,text
0,2025852395645988923,2026-02-23 14:07:00,19,FREE GOLD (XAUUSD) TRADING SIGNALS! https://t...
1,2025852394278383923,2026-02-23 14:07:00,87,$btc still needs that capitulation leg imo #b...
2,2025852386829627721,2026-02-23 14:07:00,1041,Market doesn’t care about your bias. It hunts ...
3,2025852387441689003,2026-02-23 14:07:00,2432,LOL gravedancing BTC during your come-up is no...
4,2025852384140783880,2026-02-23 14:07:00,28,🚨 FREE GOLD (XAUUSD) TRADING SIGNALS! 🚨 https:...
...,...,...,...,...
738,2025862831992885529,2026-02-23 14:48:00,86,🚨#XAUUSD #GOLD free Signal. Join Free Signals💸...
739,2025862828213637307,2026-02-23 14:48:00,423,"@RossKempsell GM Sir, good luck for the week a..."
740,2025862827819381122,2026-02-23 14:48:00,722,@DividendDrip Just the standard basket for me....
741,2025862827689357349,2026-02-23 14:48:00,41,@mamboitaliano__ Trump and his family committe...
